# Análise Raw - ecommerce_rastreamento_entregas

Objetivo:
- Ler o CSV original da camada Raw.
- Validar estrutura básica.
- Entender volume, colunas, período dos dados e distribuição dos status.
- Não escrever nada em Bronze, Silver, Gold ou SQL Server.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    min as spark_min,
    max as spark_max,
    to_timestamp,
    year,
    month
)

SOURCE_FILE = "ecommerce_rastreamento_entregas.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false"
}

EXPECTED_COLUMNS = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "codigo_rastreio",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "observacao"
]

adls_options = get_adls_options()

print(f"Arquivo Raw analisado: {SOURCE_PATH}")

In [0]:
df_raw = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

validation_result = validate_required_columns(df_raw, EXPECTED_COLUMNS)

print(validation_result["message"])
print(f"Colunas extras encontradas: {validation_result['unexpected_columns']}")

total_linhas = df_raw.count()

print(f"Total de linhas lidas da Raw: {total_linhas}")
print(f"Total de colunas: {len(df_raw.columns)}")

df_raw.printSchema()

display(df_raw.limit(10))

In [0]:
df_analysis = df_raw.withColumn(
    "dt_evento_ts",
    to_timestamp(col("dt_evento"))
)

df_resumo_geral = df_analysis.select(
    count("*").alias("total_linhas"),
    countDistinct("id_rastreamento").alias("ids_rastreamento_distintos"),
    countDistinct("id_pedido_ecommerce").alias("pedidos_distintos"),
    spark_min("dt_evento_ts").alias("dt_evento_mais_antiga"),
    spark_max("dt_evento_ts").alias("dt_evento_mais_recente")
)

display(df_resumo_geral)

df_qualidade_data = df_analysis.select(
    count("*").alias("total_linhas"),
    count(col("dt_evento")).alias("dt_evento_preenchida"),
    count(col("dt_evento_ts")).alias("dt_evento_convertida")
)

display(df_qualidade_data)

df_status = (
    df_analysis
    .groupBy("status_entrega")
    .count()
    .orderBy(col("count").desc())
)

display(df_status)

df_volume_mensal = (
    df_analysis
    .withColumn("ano", year(col("dt_evento_ts")))
    .withColumn("mes", month(col("dt_evento_ts")))
    .groupBy("ano", "mes")
    .count()
    .orderBy("ano", "mes")
)

display(df_volume_mensal)

In [0]:
BACKUP_FILE = "ecommerce_rastreamento_entregas_backup_20260618_195313.csv"
BACKUP_PATH = f"{RAW_BATCH_PATH}{BACKUP_FILE}"

print(f"Arquivo backup analisado: {BACKUP_PATH}")

df_backup = read_source_csv(
    spark=spark,
    source_path=BACKUP_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

validation_backup = validate_required_columns(df_backup, EXPECTED_COLUMNS)

print(validation_backup["message"])
print(f"Colunas extras encontradas no backup: {validation_backup['unexpected_columns']}")
print(f"Total de linhas backup: {df_backup.count()}")

df_backup.printSchema()

display(df_backup.limit(10))

In [0]:
df_backup_analysis = df_backup.withColumn(
    "dt_evento_ts",
    to_timestamp(col("dt_evento"))
)

df_resumo_backup = df_backup_analysis.select(
    count("*").alias("total_linhas"),
    countDistinct("id_rastreamento").alias("ids_rastreamento_distintos"),
    countDistinct("id_pedido_ecommerce").alias("pedidos_distintos"),
    spark_min("dt_evento_ts").alias("dt_evento_mais_antiga"),
    spark_max("dt_evento_ts").alias("dt_evento_mais_recente")
)

display(df_resumo_backup)

df_qualidade_data_backup = df_backup_analysis.select(
    count("*").alias("total_linhas"),
    count(col("dt_evento")).alias("dt_evento_preenchida"),
    count(col("dt_evento_ts")).alias("dt_evento_convertida")
)

display(df_qualidade_data_backup)

In [0]:
df_resumo_atual = df_analysis.select(
    count("*").alias("linhas_atual"),
    countDistinct("id_rastreamento").alias("ids_atual"),
    countDistinct("id_pedido_ecommerce").alias("pedidos_atual"),
    spark_min("dt_evento_ts").alias("data_min_atual"),
    spark_max("dt_evento_ts").alias("data_max_atual")
)

df_resumo_backup_comp = df_backup_analysis.select(
    count("*").alias("linhas_backup"),
    countDistinct("id_rastreamento").alias("ids_backup"),
    countDistinct("id_pedido_ecommerce").alias("pedidos_backup"),
    spark_min("dt_evento_ts").alias("data_min_backup"),
    spark_max("dt_evento_ts").alias("data_max_backup")
)

display(df_resumo_atual.crossJoin(df_resumo_backup_comp))

In [0]:
df_status_atual = (
    df_analysis
    .groupBy("status_entrega")
    .count()
    .withColumnRenamed("count", "qtd_atual")
)

df_status_backup = (
    df_backup_analysis
    .groupBy("status_entrega")
    .count()
    .withColumnRenamed("count", "qtd_backup")
)

df_comparativo_status = (
    df_status_atual
    .join(df_status_backup, on="status_entrega", how="full")
    .fillna(0)
    .withColumn("diferenca_atual_menos_backup", col("qtd_atual") - col("qtd_backup"))
    .orderBy(col("qtd_atual").desc())
)

display(df_comparativo_status)

In [0]:
df_ids_atual = df_analysis.select("id_rastreamento").dropDuplicates()
df_ids_backup = df_backup_analysis.select("id_rastreamento").dropDuplicates()

ids_no_atual_nao_no_backup = df_ids_atual.join(
    df_ids_backup,
    on="id_rastreamento",
    how="left_anti"
).count()

ids_no_backup_nao_no_atual = df_ids_backup.join(
    df_ids_atual,
    on="id_rastreamento",
    how="left_anti"
).count()

print(f"IDs que existem no arquivo atual, mas não no backup: {ids_no_atual_nao_no_backup}")
print(f"IDs que existem no backup, mas não no arquivo atual: {ids_no_backup_nao_no_atual}")

In [0]:
from pyspark.sql.functions import trim, lower

df_observacoes = (
    df_analysis
    .withColumn("observacao_normalizada", lower(trim(col("observacao"))))
    .groupBy("observacao_normalizada")
    .count()
    .orderBy(col("count").desc())
)

display(df_observacoes)